# CERRA - Catchment Time series
***

***Author:** Javier Díez Sierra, Chus Casado Rodríguez*<br>
***Date:** 24-08-2026*<br>

**Introduction:**<br>

This notebook computes the catchment-aggregated meteorological timeseries for CAMELS-ES for the meteorological reanalysis CERRA:

* [CERRA](https://cds.climate.copernicus.eu/datasets/reanalysis-cerra-single-levels?tab=overview) provides 3-hourly temperature and potential evapotranspiration.
* [CERRA-Land](https://cds.climate.copernicus.eu/datasets/reanalysis-cerra-land?tab=download) provides daily total precipitation at 6 am. CERRA-Land includes a precipitation model that assimilates observed data.

The data was downloaded from the Copernicus Climate Data Store using code within this repository ([`ocab.cerra.download.py`](../../src/ocab/cerra/download.py)). The time series start at September 1 1984 and end at August 31 2025. To speep up the computation of catchment-aggregations the CERRA dataset was converted into Zarr format in another [notebook](../meteo/CERRA_zarrify.ipynb).

In [ ]:
from tqdm.auto import tqdm

import geopandas as gpd
import xarray as xr
import logging
logger = logging.getLogger(__name__)

from ocab.config import Config
from ocab.basins.stats import read_data, basin_statistics
# from ocab.utils.geography import compute_pixel_area

## Configuration

In [ ]:
# dataset configuration
cfg = Config('./config_CAMELS_v200.yml')

# basins shapefile
basins_file = cfg.path_dataset / 'preprocessing' / 'basins' / 'output' / 'stations_basins_3sec.geojson'

# meteorology
meteo = 'CERRA'
cfg.path_meteo = cfg.path_meteo.parent.parent / meteo / 'Iberia'
zarr_store = cfg.path_meteo / f'{meteo}_1984-2025.zarr'

# output
path_out = cfg.path_dataset / 'preprocessing' / 'timeseries' / 'meteo' / meteo / 'test'
overwrite = False

## Data

In [4]:
# load basins shapefile
if basins_file.is_file():
    basins = gpd.read_file(basins_file).set_index('ID')
else:
    logger.error(f"Basins file doesn't exist: {basins_file}")
print(f'No. basins: {len(basins)}')

No. basins: 1116


In [24]:
# load meteorological data
if zarr_store.is_dir():
    data = read_data(zarr_store)
    print(f"{data.nbytes / 1e9:.2f} GB")
else:
    logger.error(f"Zarr store doesn't exist: {zarr_store}")

# rename variables
data = data.rename_vars({'totprec':'precipitation'})

# set spatial dimensions
data = data.rio.write_crs('epsg:4326')
data = data.rio.set_spatial_dims(x_dim='lon', y_dim='lat')

15.90 GB


In [ ]:
# # ensure CRS and spatial dimensions
# data = data.rio.set_spatial_dims(x_dim='lon', y_dim='lat')
# # for var in data.data_vars:
# #     data[var].rio.write_crs(data.rio.crs, inplace=True)
# #     data[var].rio.set_spatial_dims(x_dim="lon", y_dim="lat", inplace=True)
# #     print(var, da.rio.crs, da.rio.x_dim, da.rio.y_dim, sep='\t')
# for var, da in data.items():
#     da = da.rio.write_crs(data.rio.crs)
#     da = da.rio.set_spatial_dims(x_dim='lon', y_dim='lat')
#     print(var, da.rio.crs, da.rio.x_dim, da.rio.y_dim, sep='\t')
#     data[var] = da
# print('\ndataset', data.rio.crs, data.rio.x_dim, data.rio.y_dim, sep='\t')

### Pixel Area

The area of each pixel in the spatial grid would be necessary to compute the weighted basin statistics, i.e., taking into account that pixel area varies with latitude.

In [ ]:
# pixarea = compute_pixel_area(data, proj='wgs_1984').compute()
# # pixarea = pixarea.rio.set_spatial_dims(x_dim='lon', y_dim='lat')

## Basin Statistics

### Precipitation

In [26]:
# compute statistics
pcp_ts = basin_statistics(
    data=data['precipitation'],
    basins=basins,
    statistic=['mean', 'max', 'std'],
    # weight=pixarea
)

basins:   0%|          | 0/1116 [00:00<?, ?it/s]

In [27]:

# compute fraction area covered by precipitation
pcp_thr = 1 # mm
occurrence = xr.where(data['precipitation'] > pcp_thr, 1.0, 0.0)
occurrence = occurrence.rio.write_crs(data.rio.crs)
occurrence_ts = basin_statistics(
    data=occurrence,
    basins=basins,
    statistic=['count', 'sum'],
)
pcp_ts['precipitation_frac'] = occurrence_ts['precipitation_sum'] / occurrence_ts['precipitation_count']

basins:   0%|          | 0/1116 [00:00<?, ?it/s]

### Temperature & PET

In [29]:
# compute statistics
vars = ['avgtemp', 'mintemp', 'maxtemp', 'e0']
tmp_ts = basin_statistics(
    data=data[vars],
    basins=basins,
    statistic='mean',
    # weight=pixarea
)

# compute diurnal temperature range
tmp_ts['rngtemp_mean'] = tmp_ts['maxtemp_mean'] - tmp_ts['mintemp_mean']

basins:   0%|          | 0/1116 [00:00<?, ?it/s]

## Export

In [30]:
# combine variables
meteo_ts = xr.merge([pcp_ts, tmp_ts], compat='no_conflicts')

# export as Parquet files
path_out.mkdir(exist_ok=True)
for ID in tqdm(basins.index, desc='basins'):        
    fileout = path_out / f'{ID}.parquet'
    if fileout.exists() and  not overwrite:
        logger.info(f'Output file {fileout} already exists. Moving forward to the next basin')
        continue
    df = meteo_ts.sel(id=[ID]).to_dataframe()
    df.drop(['crs', 'spatial_ref', 'wgs_1984', 'rotated_pole', 'id'], axis=1, errors='ignore', inplace=True)
    df.to_parquet(fileout)

basins:   0%|          | 0/1116 [00:00<?, ?it/s]